# Imports and Device Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import models
import cv2
import numpy as np
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

# Custom Dataset Class
This class handles high-resolution MRI scans. We resize them to $224 \times 224$ to fit AlexNet's requirements.

In [ ]:
class BrainMRIDataset(Dataset):
    def __init__(self, image_paths, image_size=(224, 224)):
        self.image_paths = image_paths
        self.image_size = image_size
        
        # Extract folder names as labels
        self.labels = [Path(p).parent.name for p in image_paths]
        self.unique_labels = sorted(list(set(self.labels)))
        self.class_to_idx = {cls: i for i, cls in enumerate(self.unique_labels)}

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Load and convert to RGB (AlexNet expectation)
        img = cv2.imread(self.image_paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, self.image_size)
        
        # Intensity Normalization & ImageNet Standard Normalization
        img = img.astype(np.float32) / 255.0
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = (img - mean) / std
        
        # PyTorch expects (Channels, Height, Width)
        img = np.transpose(img, (2, 0, 1))
        
        label = self.class_to_idx[self.labels[idx]]
        return torch.from_numpy(img).float(), torch.tensor(label).long()

# Data Augmentation and Loaders
We use Data Augmentation (Rotation and Flips) to handle the limited size of medical datasets and improve model robustness.

In [ ]:
# Using the path structure found in your environment
base_path = Path('/kaggle/input/brain-tumor-mri-dataset')
all_paths = [str(p) for p in base_path.rglob('*.jpg')]

if len(all_paths) == 0:
    # Backup path check
    base_path = Path('/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset')
    all_paths = [str(p) for p in base_path.rglob('*.jpg')]

dataset = BrainMRIDataset(all_paths)

# Split logic
train_sz = int(0.8 * len(dataset))
val_sz = int(0.1 * len(dataset))
test_sz = len(dataset) - train_sz - val_sz

train_ds, val_ds, test_ds = random_split(dataset, [train_sz, val_sz, test_sz])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

print(f"Dataset initialized with {len(all_paths)} images.")
print(f"Classes found: {dataset.unique_labels}")

# AlexNet with Dropout
We modify the classifier to add Dropout for regularization, as requested in your scenario questions.

In [ ]:
class MedicalAlexNet(nn.Module):
    def __init__(self, num_classes):
        super(MedicalAlexNet, self).__init__()
        # Using pre-trained weights for Transfer Learning
        self.model = models.alexnet(weights='DEFAULT')
        
        # Scenario 3: Custom Classifier with Dropout
        self.model.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        return self.model(x)

model = MedicalAlexNet(num_classes=len(dataset.unique_labels)).to(device)

# Training Loop

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

for epoch in range(12):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    # Simple Validation check per epoch
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            
    print(f"Epoch {epoch+1} - Loss: {running_loss/len(train_loader):.4f} - Val Acc: {100*correct/len(val_ds):.2f}%")

# Medical Performance Evaluation
For MRI detection, accuracy isn't enough—we need a Confusion Matrix to see if we are missing any specific tumor types.

In [ ]:
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=dataset.unique_labels))

# Plotting Confusion Matrix
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=dataset.unique_labels, yticklabels=dataset.unique_labels)
plt.title('Confusion Matrix - Brain Tumor Detection')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()